In [47]:
fl  = 'R1233zd(E)'
cpw = 4180.0
T_DC_out = 60 + 273.15 # Water temp. coming out of DC (K)
m_w = 28.7 # Water loop flow rate for DC cooling (kg/s) 
PP, SH, SC = 5.0, 4.0, 0 # Pinch (K), superheat (K), subcool (K)
eta_t, eta_p = 0.8, 0.75 # Efficiencies of ORC turbine and pump

def orc(T_evap, T_cond):
    P_e, P_c = PS('P','T',T_evap,'Q',1,fl), PS('P','T',T_cond,'Q',1,fl) # Gives evaporator and condenser pressures - check if this should be 0 for saturated liquid

    # 1: pump inlet (sat. liquid at T_cond, minus subcooling)
    T1 = T_cond - SC # Pump inlet temp.
    h1 = PS('H','T',T1,'Q',0,fl);  s1 = PS('S','T',T1,'Q',0,fl) # Enthalpy and entropy at pump inlet
    
    # 2: pump exit
    h2_ideal = PS('H','P',P_e,'S',s1,fl) # Calculates the enthalpy at pump exit if it were isentropic (pump just increases pressure from P_c to P_e and entropy stays the same)
    h2 = h1 + (h2_ideal - h1)/eta_p # Calculates the actual enthalpy at pump exit, accounting for pump efficiency (note, isentropic efficiency is used)
    
    # 3: WF liquid preheated to evaporation temperature
    h3 = PS('H','T',T_evap,'Q',0,fl)  # Enthalpy of saturated liquid at evaporator temperature
    
    # 4: turbine inlet
    h4 = PS('H','T',T_evap+SH,'P',P_e,fl);  s4 = PS('S','T',T_evap+SH,'P',P_e,fl) # Calculates the enthalpy and entropy at turbine inlet (superheated vapor)
    
    # 5: turbine outlet
    h5_ideal = PS('H','P',P_c,'S',s4,fl) # Calculates the enthalpy at turbine exit if it were isentropic (turbine just decreases pressure from P_e to P_c and entropy stays the same)
    h5   = h4 - eta_t*(h4 - h5_ideal) # Calculates the actual enthalpy at turbine exit, accounting for turbine efficiency (note, isentropic efficiency is used)

    # pinch at bubble point: water temp there = T_evap + PP
    Q_boil = m_w*cpw*(T_DC_out - (T_evap+PP)) # Heat for boiling + superheat in kW (note the evaporator is counter-flow, so water first superheats + boils the working fluid, then preheats it)
    m_wf = Q_boil/(h4 - h3) # mass flow rate of working fluid in the ORC loop
    Q_in = m_wf*(h4 - h2) # Heat delivered to the ORC working fluid in the evaporator in kW
    T_w_out_evap = T_DC_out - Q_in/(m_w*cpw) # check To >= T_evap-ish and cold-end pinch

    W_t, W_p = m_wf*(h4-h5), m_wf*(h2-h1)    # Work output from turbine and work input to pump
    Q_cond = m_wf*(h5 - h1) # Heat rejected to the condenser
    return dict(m_wf=m_wf, Q_in=Q_in, T_w_out_evap=T_w_out_evap, W_net=W_t-W_p, Q_cond=Q_cond, ORC_efficiency=100*(W_t-W_p)/Q_in)

In [48]:
import numpy as np
from CoolProp.CoolProp import PropsSI as PS
from CoolProp.HumidAirProp import HAPropsSI

def calculate_system_performance(T_evap_C, T_cond_C, T_db_amb, T_wb_amb, P_atm):
    # Convert to Kelvin
    T_evap = T_evap_C + 273.15
    T_cond = T_cond_C + 273.15
    
    # 1. Run ORC Model
    res = orc(T_evap, T_cond)  # Assumes your orc() function is already defined
    W_net_orc = res['W_net']    # Net power of ORC block (Turbine - WF Pump) (W)
    Q_cond = res['Q_cond']      # Condenser heat rejection (W)
    T_w_out_evap = res['T_w_out_evap']              # Water temperature leaving evaporator (K)
    ORC_actual_efficiency = res['ORC_efficiency']
    
    # 2. Calculate pump powers at condenser and DC

    # Calculate required heat removal by the trim cooler
    T_w_out_evap = orc(T_evap, T_cond)['T_w_out_evap']
    T_DC_return = 40 + 273.15  # T of water flowing back into DC
    Q_trim = m_w * cpw * (T_w_out_evap - T_DC_return)

    # Approximate density of water at condenser temperature
    rho_w = 1000 # kg/m^3 (approximate density of water) - should this be mmore specific?

    # Calculate volume flow rate of water in condensor loop (not mass flow rate)
    V_w_cond = Q_cond/(rho_w*cpw*(T_range))

    # Calculate volume flow rate of water in DC water loop - based on established DC water flow rate
    V_w_DC = m_w/rho_w

    # Define pressure drop in both condenser and DC loop (Pa)
    dp = 100e3

    # Define pump motor and impeller efficiencies for condenser and DC loop
    eta_m, eta_imp = 0.9, 0.78 

    # Calculate the work required to pump the water through the condenser loop
    W_pump_cond = V_w_cond * dp / (eta_m * eta_imp)
    

    # Calculate the work required to pump water through DC loop
    W_pump_DC = V_w_DC * dp / (eta_m * eta_imp)

    # 3. Calculate the fan powers from the water towers at condenser and DC
    from CoolProp.HumidAirProp import HAPropsSI

    # Define ambient air conditions at the inlet - assume dry bulb temp.
    RH_in = 0.50 # Assume 50% relative humidity for ambient air (check from where)
    P_atm = 94100.0 # Atmospheric pressure (Pa) - potentially revise to the specific pressure of las vegas?
    RH_out = 1.0 # Saturated air leaving the wet deck - assume 100% saturated

    dp_air = 120.0          # Pressure drop across fill/eliminators (Pa) - find justification for this
    eta_fan = 0.65          # Fan efficiency - justify
    eta_motor_fan = 0.90    # Motor efficiency - justify

    # Calculates mass density of inlet air directly (kg/m^3)
    rho_air = PS('D', 'T', T_db_amb, 'P', P_atm, 'HEOS::Air')

    # Calculate inlet air specific enthalpy (J / kg of dry air)
    h_air_in_cond = HAPropsSI('H', 'T', T_db_amb, 'R', RH_in, 'P', P_atm)
    h_air_in_DC = HAPropsSI('H', 'T', T_db_amb, 'R', RH_in, 'P', P_atm)

    # Define exit air conditions
    # Exit air is typically assumed saturated (100% RH) and warmed close to the incoming water temp
    T_w_WT_cond = T_wb_amb + T_approach + T_range # Hot water T leaving condensor
    T_air_out_cond = T_w_WT_cond - 2.0 # Exit air temperature approaches water-in temperature (find justification for this 2K)

    T_air_out_DC = T_w_out_evap - 2.0 # Exit air temperature approaches water-in temperature (find justification for this 2K)

    # Calculate exit air specific enthalpy (J / kg of dry air)
    h_air_out_cond = HAPropsSI('H', 'T', T_air_out_cond, 'R', RH_out, 'P', P_atm)
    h_air_out_DC = HAPropsSI('H', 'T', T_air_out_DC, 'R', RH_out, 'P', P_atm)

    # 3. Solve for required air mass flow rate (m_air)
    # Q_cond is in Watts
    m_air_cond = Q_cond / (h_air_out_cond - h_air_in_cond)

    m_air_DC = Q_trim / (h_air_out_DC - h_air_in_DC)

    # Calculate volume flow rate
    V_air_cond = m_air_cond / rho_air # Volume flow rate (m^3/s)
    V_air_DC = m_air_DC / rho_air # Volume flow rate (m^3/s)

    W_ct_fan_cond = (V_air_cond * dp_air) / (eta_fan * eta_motor_fan)

    W_ct_fan_DC = (V_air_DC * dp_air) / (eta_fan * eta_motor_fan)

    # 4. Calculating Carnot efficiency of the OCR loop
    eta_Carnot_ORC = (1 - T_cond/T_evap)*100

    # Calculating Carnot efficiency from the broad reservoir limit
    eta_Carnot_broad = (1 - T_wb_amb/T_DC_out)*100

    
    # 5. Total System Net Power Output
    total_parasitics = W_pump_cond + W_pump_DC + W_ct_fan_cond + W_ct_fan_DC
    W_net_system = W_net_orc - total_parasitics
    
    return {
        'W_net_orc': W_net_orc,
        'W_net_system': W_net_system,
        'total_parasitics': total_parasitics,
        'ORC_actual_efficiency': ORC_actual_efficiency,
        'ORC_carnot_efficiency': eta_Carnot_ORC,
        'broad_carnot_efficiency': eta_Carnot_broad
    }

In [49]:
T_PP_condensor = 5
T_approach = 4
T_range = 5
T_cond_C = T_wb_amb + T_approach + T_range + T_PP_condensor - 273.15


T_evap_C = 30
T_db_amb = 20 + 273.15
T_wb_amb = 10 + 273.15
P_atm = 94100

print(calculate_system_performance(T_evap_C, T_cond_C, T_db_amb, T_wb_amb, P_atm))

{'W_net_orc': 48128.2817276693, 'W_net_system': -29766.511228729512, 'total_parasitics': 77894.79295639881, 'ORC_actual_efficiency': 1.5472481276582473, 'ORC_carnot_efficiency': 1.9792182088075205, 'broad_carnot_efficiency': 15.008254539997}


In [66]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

# 1. Load or define your weather bin data
weather_df = pd.DataFrame({
    'T_db': [-2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46],
    'T_wb': [-11.6, -6.7, -3.9, -2.1, -0.1, 2.2, 3.9, 4.9, 6.0, 7.3, 8.5, 10.0, 11.5, 12.7, 13.6, 14.6, 15.7, 17.0, 18.3, 18.9, 19.7, 19.6, 20.4, 21.5, 21.9],
    'hours': [14, 50, 126, 215, 310, 471, 528, 527, 564, 544, 516, 543, 525, 570, 535, 540, 470, 370, 332, 271, 250, 212, 178, 93, 6]
})

P_atm = 94100.0  # Atmospheric pressure (Pa)
T_approach = 4.0 # Approach temperature (K)
T_range = 5.0    # Range temperature (K)

results_list = []

for index, row in weather_df.iterrows():
    T_db_C = row['T_db']
    T_wb_C = row['T_wb']
    bin_hours = row['hours']
    
    T_db_amb = T_db_C + 273.15
    T_wb_amb = T_wb_C + 273.15
    
    # Dynamically calculate condenser temperature based on bin wet-bulb temperature (°C)
    T_cond_C = T_wb_C + T_approach + T_range

    # R1233zd(E) and cooling tower dynamics struggle or become unphysical at deep sub-zero condenser temps
    if T_cond_C < 2.0:
        results_list.append({
            'T_db_bin': T_db_C,
            'T_wb_bin': T_wb_C,
            'T_cond_C': T_cond_C,
            'TMY_hours': bin_hours,
            'Optimal_T_evap_C': np.nan,
            'W_net_orc_kW': 0.0,
            'W_net_system_kW': 0.0,
            'Total_Parasitics_kW': 0.0,
            'Annual_Net_Energy_MWh': 0.0
        })
        continue
    
    # Define objective function to maximize net system power
    def objective(T_evap_C):
        try:
            res = calculate_system_performance(T_evap_C, T_cond_C, T_db_amb, T_wb_amb, P_atm)
            net_power_W = res['W_net_system']
            if np.isnan(net_power_W) or res['total_parasitics'] > res['W_net_orc']: # No point installing the ORC if parasitics are larger
                return 0.0
            return -net_power_W # Negative so minimising the objective maximises net power
        except Exception:
            return 0.0


    # Optimize T_evap for this specific bin's conditions
    opt_result = minimize_scalar(objective, bounds=(30, 55.0), method='bounded')
    
    best_T_evap = opt_result.x
    max_net_power_W = -opt_result.fun
    
    # Run final evaluation with optimal T_evap
    best_metrics = calculate_system_performance(best_T_evap, T_cond_C, T_db_amb, T_wb_amb, P_atm)
    
    # Annual energy totals (MWh)
    annual_net_system_MWh = (best_metrics['W_net_system'] * bin_hours) / 1e6
    
    results_list.append({
        'T_db_bin': T_db_C,
        'T_wb_bin': T_wb_C,
        'T_cond_C': T_cond_C,
        'TMY_hours': bin_hours,
        'Optimal_T_evap_C': best_T_evap,
        'W_net_orc_kW': best_metrics['W_net_orc'] / 1e3,
        'W_net_system_kW': best_metrics['W_net_system'] / 1e3,
        'Total_Parasitics_kW': best_metrics['total_parasitics'] / 1e3,
        'Annual_Net_Energy_MWh': annual_net_system_MWh
    })

# 2. Final DataFrame containing all optimized data per bin temperature pair
bin_results_df = pd.DataFrame(results_list)

# Preview the DataFrame that will make your final plotting effortless
print(bin_results_df[['T_db_bin', 'T_wb_bin', 'T_cond_C', 'Optimal_T_evap_C', 'W_net_system_kW']].head(10))

   T_db_bin  T_wb_bin  T_cond_C  Optimal_T_evap_C  W_net_system_kW
0      -2.0     -11.6      -2.6               NaN         0.000000
1       0.0      -6.7       2.3         35.017902       122.537297
2       2.0      -3.9       5.1         34.827458       125.394187
3       4.0      -2.1       6.9         35.508653       116.333403
4       6.0      -0.1       8.9         36.139254       108.289671
5       8.0       2.2      11.2         36.794118       100.275243
6      10.0       3.9      12.9         37.530868        91.547881
7      12.0       4.9      13.9         38.298687        82.813846
8      14.0       6.0      15.0         39.111911        74.059454
9      16.0       7.3      16.3         39.910317        65.959482
